# Load and Inspect the Dataset

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the dataset (I first expanded the data for better understanding and easy analysis)
file_path = 'expanded_trade_data.csv'
df = pd.read_csv(file_path)
df.head()

,Port_ID,time,symbol,side,price,fee,feeAsset,quantity,quantityAsset,realizedProfit,realizedProfitAsset,baseAsset,qty,positionSide,activeBuy
0,3925368433214965504,2024-06-20 16:07:36,SOLUSDT,BUY,132.53700,-0.994027,USDT,1988.05500,USDT,0.0,USDT,SOL,15.0,LONG,True
1,3925368433214965504,2024-06-20 16:06:58,DOGEUSDT,BUY,0.12182,-0.279796,USDT,1398.98088,USDT,0.0,USDT,DOGE,11484.0,LONG,False
2,3925368433214965504,2024-06-20 16:06:58,DOGEUSDT,BUY,0.12182,-0.039494,USDT,197.47022,USDT,0.0,USDT,DOGE,1621.0,LONG,False
3,3925368433214965504,2024-06-20 16:06:56,DOGEUSDT,BUY,0.12182,-0.008284,USDT,16.56752,USDT,0.0,USDT,DOGE,136.0,LONG,True
4,3925368433214965504,2024-06-20 16:06:56,DOGEUSDT,BUY,0.12182,-0.046109,USDT,92.21774,USDT,0.0,USDT,DOGE,757.0,LONG,True


In [4]:
print(df.info())
print(df.head())

print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211277 entries, 0 to 211276
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Port_ID              211277 non-null  int64  
 1   time                 211277 non-null  object 
 2   symbol               211277 non-null  object 
 3   side                 211277 non-null  object 
 4   price                211277 non-null  float64
 5   fee                  211277 non-null  float64
 6   feeAsset             211277 non-null  object 
 7   quantity             211277 non-null  float64
 8   quantityAsset        211277 non-null  object 
 9   realizedProfit       211277 non-null  float64
 10  realizedProfitAsset  211277 non-null  object 
 11  baseAsset            211277 non-null  object 
 12  qty                  211277 non-null  float64
 13  positionSide         211277 non-null  object 
 14  activeBuy            211277 non-null  bool   
dtypes: bool(1), float

# Data Cleaning & Preprocessing
#### Convert time columns, handle duplicates, and ensure data types are correct.

In [5]:
# Convert 'time' column to datetime
df['time'] = pd.to_datetime(df['time'])
print(df.dtypes)

# Remove duplicates if any
df.drop_duplicates(inplace=True)



Port_ID                         int64
time                   datetime64[ns]
symbol                         object
side                           object
price                         float64
fee                           float64
feeAsset                       object
quantity                      float64
quantityAsset                  object
realizedProfit                float64
realizedProfitAsset            object
baseAsset                      object
qty                           float64
positionSide                   object
activeBuy                        bool
dtype: object


# Feature Engineering

In [6]:
# Classify trade positions
df['trade_type'] = df['side'] + "_" + df['positionSide']

# cumulative profit for each account
df['cumulative_profit'] = df.groupby('Port_ID')['realizedProfit'].cumsum()

# Sort by account and time
df.sort_values(by=['Port_ID', 'time'], inplace=True)

# Calculate Financial Metrics

In [7]:
# ROI for each account
roi = df.groupby('Port_ID').apply(lambda x: x['realizedProfit'].sum() / x['quantity'].sum()).reset_index()
roi.columns = ['Port_ID', 'ROI']

# Total PnL for each account
pnl = df.groupby('Port_ID')['realizedProfit'].sum().reset_index()
pnl.columns = ['Port_ID', 'PnL']

In [8]:
# Annualized Sharpe Ratio
def sharpe_ratio(x):
    return np.mean(x) / np.std(x) if np.std(x) != 0 else 0

sharpe = df.groupby('Port_ID')['realizedProfit'].apply(sharpe_ratio).reset_index()
sharpe.columns = ['Port_ID', 'Sharpe_Ratio']

In [9]:
def max_drawdown(profit_series):
    roll_max = profit_series.cummax()
    drawdown = (profit_series - roll_max) / roll_max
    return drawdown.min()

mdd = df.groupby('Port_ID')['cumulative_profit'].apply(max_drawdown).reset_index()
mdd.columns = ['Port_ID', 'MDD']


In [10]:
# Win rate
df['win'] = df['realizedProfit'] > 0
win_rate = df.groupby('Port_ID')['win'].mean().reset_index()
win_rate.columns = ['Port_ID', 'Win_Rate']

# Count total and winning positions
total_positions = df.groupby('Port_ID')['Port_ID'].count().reset_index(name='Total_Positions')
win_positions = df.groupby('Port_ID')['win'].sum().reset_index()
win_positions.columns = ['Port_ID', 'Win_Positions']

# Ranking Accounts

In [11]:
# Merge all metrics
metrics_df = roi.merge(pnl, on='Port_ID').merge(sharpe, on='Port_ID').merge(mdd, on='Port_ID')
metrics_df = metrics_df.merge(win_rate, on='Port_ID').merge(total_positions, on='Port_ID').merge(win_positions, on='Port_ID')

weights = {'ROI': 0.3, 'PnL': 0.2, 'Sharpe_Ratio': 0.2, 'MDD': -0.2, 'Win_Rate': 0.1, 'Total_Positions': 0.1}

metrics_df['Score'] = (
    weights['ROI'] * metrics_df['ROI'] +
    weights['PnL'] * metrics_df['PnL'] +
    weights['Sharpe_Ratio'] * metrics_df['Sharpe_Ratio'] +
    weights['MDD'] * metrics_df['MDD'] +  
    weights['Win_Rate'] * metrics_df['Win_Rate'] +
    weights['Total_Positions'] * metrics_df['Total_Positions']
)

# Rank accounts
metrics_df.sort_values(by='Score', ascending=False, inplace=True)

# Export Results

In [12]:
top_20 = metrics_df.head(20)
top_20.to_csv("top_20_accounts.csv", index=False)


In [13]:
report = f"""
### Analysis Report

#### **Methodology**
1. Cleaned and preprocessed data.
2. Engineered features like cumulative profit and trade type.
3. Calculated financial metrics: ROI, PnL, Sharpe Ratio, MDD, Win Rate, Total Positions.
4. Ranked accounts based on weighted scores.

#### **Findings**
- Top 20 accounts were identified.
- The best-performing account had an ROI of {top_20.iloc[0]['ROI']:.2f} and a Sharpe Ratio of {top_20.iloc[0]['Sharpe_Ratio']:.2f}.
- Maximum Drawdown indicates risk management effectiveness.
"""

# Save report
with open("report.txt", "w") as f:
    f.write(report)

print("Analysis complete! Top 20 accounts saved in 'top_20_accounts.csv' and report generated.")


Analysis complete! Top 20 accounts saved in 'top_20_accounts.csv' and report generated.
